In [1]:
import jax
import jax.numpy as jnp

from furax._base.blocks import BlockDiagonalOperator, BlockRowOperator
from furax._base.core import HomothetyOperator, IdentityOperator
from furax.landscapes import StokesPyTree, ValidStokesType
from furax.tree import as_structure
from furax.operators.sed import CMBOperator, DustOperator, SynchrotronOperator
import operator
from math import prod
from itertools import product

In [2]:
nside = 4
npixel = 12 * nside**2
stokes_type: ValidStokesType = 'IQU'
frequency_count = 10

In [3]:
comp_keys = jax.random.split(jax.random.PRNGKey(0), 3)
stokes_cls = StokesPyTree.class_for(stokes_type)
frequencies = jnp.logspace(10, 12, num=frequency_count)
sky = {
    'cmb': stokes_cls.normal(comp_keys[0], (npixel, )),
    'dust': stokes_cls.normal(comp_keys[1], (npixel, )),
    'synchrotron': stokes_cls.normal(comp_keys[2], (npixel, )),
}

In [4]:
def make_mixing_matrix_operator(params, in_structure):
    cmb = CMBOperator(frequencies, in_structure=in_structure)
    dust = DustOperator(
        frequencies,
        temperature=params['temp_dust'],
        beta=params['beta_dust'],
        in_structure=in_structure,
    )
    synchrotron = SynchrotronOperator(frequencies,
                                      beta_pl=params['beta_pl'],
                                      in_structure=in_structure)
    sed = BlockDiagonalOperator({
        'cmb': cmb,
        'dust': dust,
        'synchrotron': synchrotron
    })
    integ = BlockRowOperator({
        component:
        IdentityOperator(sed.blocks[component].out_structure())
        for component in sed.blocks
    })
    return (integ @ sed).reduce()

In [6]:
from furax.landscapes import HealpixLandscape
best_params = {
    'temp_dust': 22.,
    'beta_dust': 1.54,
    'beta_pl': -3.0,
}
in_structure = HealpixLandscape(nside, stokes_type).structure
A = make_mixing_matrix_operator(best_params,in_structure=in_structure)
d = A(sky)

In [7]:
invN = HomothetyOperator(jnp.ones(1), _in_structure=d.structure)
DND = invN(d) @ d

@jax.jit
def likelihood(params, d):
    A = make_mixing_matrix_operator(params,in_structure=in_structure)

    number_of_components = len(A.in_structure())
    pixels_freq = prod(A.out_structure().shape)

    x = (A.T @ invN)(d)
    l = jax.tree.map(lambda a, b: a @ b, x, (A.T @ invN @ A).I(x))
    summed_likelihood = jax.tree.reduce(operator.add, l)

    return -summed_likelihood + DND

In [10]:
likelihood(best_params, d)

Array(0., dtype=float64)

In [8]:
wrong_params = jax.tree.map(lambda x: x + jax.random.normal(jax.random.PRNGKey(0)), best_params)
print(f"Wrong parameters grad {jax.tree.reduce(max , jax.grad(likelihood)(wrong_params, d))}")
print(f"Correct parameters grad {jax.tree.reduce(max , jax.grad(likelihood)(best_params, d))}")

max_steps: 50000
Wrong parameters grad 10697.543215847583
max_steps: 50000
Correct parameters grad 7.704241203257867e-13


In [106]:
likelihood(params, d).block_until_ready()
%timeit likelihood(params, d).block_until_ready()

4.89 ms ± 531 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [107]:
jax.grad(likelihood)(params, d)['beta_pl'].block_until_ready()
%timeit jax.grad(likelihood)(params, d)['beta_pl'].block_until_ready()

14.7 ms ± 427 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [146]:
import jaxopt

param_keys = jax.random.split(jax.random.PRNGKey(1), 3)
params = {
    'beta_dust': jax.random.uniform(param_keys[0], (1, ),
                                    minval=1.0,
                                    maxval=3.0),
    'temp_dust': jax.random.uniform(param_keys[1], (1, ), minval=15., maxval=50.),
    'beta_pl': jax.random.uniform(param_keys[2], (1, ), minval=-6.,
                                  maxval=-1.),
}

solver = jaxopt.ScipyMinimize(fun=likelihood, method='TNC',maxiter=1000 , jit=True ,tol=1e-10)
%timeit  solver.run(params, d).params['beta_pl'].block_until_ready()

/home/wassim/micromamba/envs/cmb/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),
/home/wassim/micromamba/envs/cmb/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


311 ms ± 14.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [147]:
optimized_params = solver.run(best_params, d).params
optimized_params

{'beta_dust': Array(1.54, dtype=float64),
 'beta_pl': Array(-2.99999999, dtype=float64),
 'temp_dust': Array(22., dtype=float64)}

In [149]:
params = {
    'temp_dust': 15.,
    'beta_dust': 1.54,
    'beta_pl': -3.0,
}

optimized_params = solver.run(params, d).params
optimized_params


{'beta_dust': Array(1.56843666, dtype=float64),
 'beta_pl': Array(-3.00000001, dtype=float64),
 'temp_dust': Array(22.23851438, dtype=float64)}

In [150]:
solver = jaxopt.ScipyMinimize(fun=likelihood, method='Nelder-Mead',maxiter=1000 , jit=True ,tol=1e-10)
params = {
    'temp_dust': 15.,
    'beta_dust': 3.0,
    'beta_pl': -3.0,
}

optimized_params = solver.run(params, d).params
optimized_params


/home/wassim/micromamba/envs/cmb/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: RuntimeWarning: Method Nelder-Mead does not use gradient information (jac).
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.54060523, dtype=float64),
 'beta_pl': Array(-2.99999995, dtype=float64),
 'temp_dust': Array(22.00510306, dtype=float64)}

In [152]:
solver = jaxopt.ScipyMinimize(fun=likelihood, method='Nelder-Mead',maxiter=1000 , jit=True ,tol=1e-10)
params = {
    'temp_dust': 15.,
    'beta_dust': 1.54,
    'beta_pl': -6.0,
}

optimized_params = solver.run(params, d).params
optimized_params


/home/wassim/micromamba/envs/cmb/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: RuntimeWarning: Method Nelder-Mead does not use gradient information (jac).
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.5750274, dtype=float64),
 'beta_pl': Array(-3.00000016, dtype=float64),
 'temp_dust': Array(22.29471025, dtype=float64)}

In [154]:
solver = jaxopt.ScipyMinimize(fun=likelihood, method='Nelder-Mead',maxiter=1000 , jit=True ,tol=1e-10)
params = {
    'temp_dust': 15.,
    'beta_dust': 3.0,
    'beta_pl': -6.0,
}

optimized_params = solver.run(params, d).params
optimized_params


/home/wassim/micromamba/envs/cmb/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: RuntimeWarning: Method Nelder-Mead does not use gradient information (jac).
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(2.36053751, dtype=float64),
 'beta_pl': Array(-3.00000051, dtype=float64),
 'temp_dust': Array(28.91607031, dtype=float64)}

In [158]:
import numpyro
import numpyro.distributions as dist
import jax
import jax.numpy as jnp
from numpyro.infer import MCMC, NUTS
from furax.landscapes import HealpixLandscape
invN = HomothetyOperator(jnp.ones(1), _in_structure=d.structure)  # Assuming this is already set
in_structure = HealpixLandscape(nside, stokes_type).structure
# Function to calculate the Gaussian log-likelihood based on `d` and mixing matrix `A`
@jax.jit
def log_likelihood(params, d):
    # Create the mixing matrix operator `A` based on `params` without needing `sky`
    A = make_mixing_matrix_operator(params, in_structure=in_structure)  # Pass shape of `d` to `make_mixing_matrix_operator`
    
    # Project data using `A` and noise inverse `invN`
    x = (A.T @ invN)(d)
    l = jax.tree.map(lambda a, b: a @ b, x, (A.T @ invN @ A).I(x))  # Computes likelihood term
    summed_likelihood = jax.tree.reduce(operator.add, l)   # Summing over components
    
    return summed_likelihood  # Positive log-likelihood value

# Define the NumPyro model
def model(d):
    # Define priors for spectral parameters
    temp_dust = numpyro.sample("temp_dust", dist.Uniform(15., 50.))
    beta_dust = numpyro.sample("beta_dust", dist.Uniform(1.0, 3.0))
    beta_pl = numpyro.sample("beta_pl", dist.Uniform(-6.0, -1.0))

    # Pack parameters to use in the likelihood function
    params = {
        'temp_dust': temp_dust,
        'beta_dust': beta_dust,
        'beta_pl': beta_pl
    }

    # Compute and add the log-likelihood to the model using numpyro.factor
    numpyro.factor("log_likelihood", log_likelihood(params, d))

# Run MCMC sampling with a reduced sample count for quick testing
rng_key = jax.random.PRNGKey(1)
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=500)  # Adjust sample count for speed
mcmc.run(rng_key, d=d)
samples = mcmc.get_samples()

mcmc.print_summary()

sample: 100%|██████████| 1000/1000 [06:43<00:00,  2.48it/s, 51 steps of size 5.33e-02. acc. prob=0.95]  



                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  beta_dust      2.10      0.52      2.11      1.33      2.95    112.80      1.00
    beta_pl     -3.00      0.00     -3.00     -3.00     -3.00    554.75      1.00
  temp_dust     27.14      4.97     27.40     19.99     35.66    120.16      1.00

Number of divergences: 0


In [159]:
# Define the NumPyro model
def model(d):
    # Define priors for spectral parameters
    temp_dust = numpyro.sample("temp_dust", dist.Uniform(15., 25.))
    beta_dust = numpyro.sample("beta_dust", dist.Uniform(1.0, 2.0))
    beta_pl = numpyro.sample("beta_pl", dist.Uniform(-6.0, -1.0))

    # Pack parameters to use in the likelihood function
    params = {
        'temp_dust': temp_dust,
        'beta_dust': beta_dust,
        'beta_pl': beta_pl
    }

    # Compute and add the log-likelihood to the model using numpyro.factor
    numpyro.factor("log_likelihood", log_likelihood(params, d))

# Run MCMC sampling with a reduced sample count for quick testing
rng_key = jax.random.PRNGKey(1)
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=500)  # Adjust sample count for speed
mcmc.run(rng_key, d=d)
samples = mcmc.get_samples()

mcmc.print_summary()

sample: 100%|██████████| 1000/1000 [07:51<00:00,  2.12it/s, 63 steps of size 5.06e-02. acc. prob=0.95]  


                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  beta_dust      1.47      0.26      1.45      1.03      1.84     98.02      1.01
    beta_pl     -3.00      0.00     -3.00     -3.00     -3.00    632.91      1.00
  temp_dust     21.24      2.14     21.30     18.51     24.87    109.02      1.00

Number of divergences: 0


In [160]:
from numpyro.handlers import condition

# Fix the temperature of dust to 22.0 and beta power-law to -3.0
conditioned_model = condition(model, {
    'temp_dust': 22.0,
    'beta_pl': -3.0
})
# Run MCMC sampling with a reduced sample count for quick testing
rng_key = jax.random.PRNGKey(1)
kernel = NUTS(conditioned_model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=500)  # Adjust sample count for speed
mcmc.run(rng_key, d=d)
samples = mcmc.get_samples()

mcmc.print_summary()

sample: 100%|██████████| 1000/1000 [00:23<00:00, 42.27it/s, 3 steps of size 8.02e-01. acc. prob=0.90]



                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  beta_dust      1.54      0.19      1.54      1.23      1.87    198.12      1.00

Number of divergences: 0


In [161]:
from numpyro.handlers import condition

# Fix the temperature of dust to 22.0 and beta power-law to -3.0
conditioned_model = condition(model, {
    'beta_pl': -3.0
})
# Run MCMC sampling with a reduced sample count for quick testing
rng_key = jax.random.PRNGKey(1)
kernel = NUTS(conditioned_model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=500)  # Adjust sample count for speed
mcmc.run(rng_key, d=d)
samples = mcmc.get_samples()

mcmc.print_summary()

sample: 100%|██████████| 1000/1000 [00:41<00:00, 23.95it/s, 3 steps of size 3.80e-01. acc. prob=0.93]



                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  beta_dust      1.44      0.26      1.42      1.06      1.86    203.41      1.00
  temp_dust     21.21      2.10     21.35     17.77     24.43    185.87      1.00

Number of divergences: 0


In [162]:
from numpyro.handlers import condition

# Fix the temperature of dust to 22.0 and beta power-law to -3.0
conditioned_model = condition(model, {
    'beta_dust': 1.54,
    'beta_pl': -3.0
})
# Run MCMC sampling with a reduced sample count for quick testing
rng_key = jax.random.PRNGKey(1)
kernel = NUTS(conditioned_model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=500)  # Adjust sample count for speed
mcmc.run(rng_key, d=d)
samples = mcmc.get_samples()

mcmc.print_summary()

sample: 100%|██████████| 1000/1000 [00:24<00:00, 40.17it/s, 7 steps of size 6.92e-01. acc. prob=0.91]



                 mean       std    median      5.0%     95.0%     n_eff     r_hat
  temp_dust     21.95      1.39     21.92     20.15     24.69    130.63      1.00

Number of divergences: 0
